# Seguimiento a embarques
- V9. 2025-02-08
    - Correccion al proceso de integracion de envios a ELP
    - El Shipment transactions ya no es acumulable debido a que no se pueden eliminar duplicados
    - Se agrega proceso de integracion de Yield reports (falta determinar como visualizarlos)
    - Se agrega inicio para panel
- V8. 2025-02-06
    - El archivo korrus_Data no es obligatorio
    - Drop Zone es NULL en lugar de blanco
- V7. 2025-02-04
    - Mejoras en el manejo de errores
- V6. 2025-02-03
    - Se integra el archivo de ELP master log si esta seleccionado
    - Correcciones para la primera ejecicion del script
- V5. 2025-02-01
    - Se elimino fecha de reparacion    
- V4. 2025-01-31
    - Correcciones para evitar errores por dataframes vacios
    - Se ajustan las columnas de embarques al paso para que sean los del reporte consolidado actual
- V3. 2025-01-28
    - Se guarda el ancho de columnas
    - Se cambia el orden de algunas columnas
- V2. 2025-01-28
    - Se consolidan los archivos Master Edi, Shipped to Cust y Shipped to ELP
    - Se genera el reporte OOR
- V1. Version inicial

## Seleccionar archivos

In [3]:
# Imports
import pandas as pd
import os
import ipywidgets as widgets
from ipywidgets import  Layout,HTML
from IPython.display import display, Markdown, clear_output
import pickle
from datetime import datetime, timedelta, timezone, date
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Alignment, Font
from openpyxl.formatting.rule import CellIsRule
from tqdm.notebook import tqdm
from tkinter import Tk, filedialog as fd
import win32com.client
import warnings
import shutil
import logging

warnings.filterwarnings("ignore")
def open_file_selection(initialdir='',filter_name='*.*'):
    """
    Opens a file selection dialog and returns the selected file paths.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: A tuple of selected file paths or an empty tuple if no file is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top
    
    filetypes = (
        (filter_name,f"*{filter_name.split(' ')[0]}*.*"),
        ('All files', '*.*'),
        ('Excel', '*.xlsx'),
        ('CSV', '*.doc')
    )
    files = fd.askopenfilenames(
        filetypes=filetypes,
        initialdir=initialdir
    )
    
    # Destroy the root window after use
    root.destroy()
    
    return files
def select_directory(initialdir):
    """
    Opens a directory selection dialog and returns the selected directory path.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: The selected directory path as a string, or an empty string if no directory is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top

    # Open the directory selection dialog
    directory = fd.askdirectory(initialdir=initialdir)

    # Destroy the root window after use
    root.destroy()

    return directory

# Function to show a pop-up message
def show_popup_message(message):
    display(Markdown(f"### **{message}**"))

    

# Decorator to handle the permission error
def handle_permission_error_with_popup(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except PermissionError as e:
            if e.errno == 13:  # Permission denied error
                show_popup_message(f"Error: {e}\nFavor de cerrar el archivo.")
    return wrapper

def save_df(df, filepath, sheet_name='Sheet', index=False):
    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index)

@handle_permission_error_with_popup
def save_df_multiple(df_dict=dict(), filepath='',index=False):
    with pd.ExcelWriter(filepath) as writer:
        for key in df_dict.keys():
            df_dict[key].to_excel(writer, sheet_name=key, index=index)

@handle_permission_error_with_popup
def save_wb(wb, filepath):
    wb.save(filepath)

@handle_permission_error_with_popup
def append_sheet(df,path,sheet_name,index):
    with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index) 


def is_file_open(filepath):
    # Check if the file exists
    if not os.path.exists(filepath):
        return False  # File does not exist, so treat it as "open" for your logic

    try:
        # Try to open the file in write mode
        with open(filepath, 'a'):
            pass
        return False  # File is not open (no exception raised)
    except PermissionError:
        return True 
    
def get_path(file_selectors,selector_name):
    file_selector=[file_selector for file_selector in file_selectors if file_selector.children[0].description[0:-1]==selector_name]
    if len(file_selector)==0:
        show_popup_message(f"No se encontro el selector")
        raise SystemExit()
    file_selector=file_selector[0]
    if not file_selector.children[1].value:
        return None
    selected_path=file_selector.children[1].value.strip()
    return selected_path

def read_excel(path=None,sheet_name=0,header=0):
    with pd.ExcelFile(path) as xls:
        df=pd.read_excel(path,sheet_name=sheet_name,header=header)
    return df

def verify_selections(file_selectors):
    not_selected=[]
    for file_selector in file_selectors:
        selector=file_selector.children[0].description[0:-1]
        selected=file_selector.children[1].value.strip()
        if not os.path.exists(selected):
            file_selector.children[1].value='Not selected'
            selected='Not selected'
        if selector not in mandatory_selectors:
            continue
        if selected=='Not selected':
            not_selected.append(selector)
            continue

    if len(not_selected)>0:
        show_popup_message(f'Favor de seleccionar los siguientes archivos:{not_selected}')
        raise SystemExit()

def close_xl_if_open(path):
    if is_file_open(path):
        try:
            excel = win32com.client.Dispatch("Excel.Application")
            workbook = excel.Workbooks(path)
            workbook.Save()
            workbook.Close()
        except:
            show_popup_message(f"Cerrar el archivo: {path}")
            raise SystemExit()               

def format_dates(df, date_cols=[]):
    for col in date_cols:
        df[col]=pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

wb_preserve_info = {
'OOR':{
    'user_cols':['Comments'],
    'keys':['PurchaseOrder','LineNumber','ProductServiceID']
    },
'Shipped to Cust':{
    'user_cols':['Comments'],
    'keys':['PO','Modelo']
    },  
'Shipped to Elp':{
    'user_cols':['Comments'],
    'keys':['PO','Modelo']
    },    
}

def  preserve_info(new_dict_wb,path_old_wb):
    """
    Preserve the data in coluns from an existing workbook in a dict dataframe
    """
    dict_wb = read_excel(path_old_wb,sheet_name=None)
    for wb_key in dict_wb.keys():      
        if not wb_key in wb_preserve_info.keys():
            continue
        user_cols = [item for item in wb_preserve_info[wb_key]['user_cols'] if item in dict_wb[wb_key].columns]
        if not user_cols:
            continue
        df=dict_wb[wb_key][wb_preserve_info[wb_key]['keys']+user_cols]
        df.drop_duplicates(inplace=True)
        new_dict_wb[wb_key]=new_dict_wb[wb_key].merge(df,on=wb_preserve_info[wb_key]['keys'],how='left') 
    return new_dict_wb   

def fillna_by_col(df,columns=[],fill_value=0):
    for col in columns:
        df[col]=df[col].astype(str)
        df[col]=df[col].fillna(fill_value)
    return df

def append_df_to_excel(df_new=pd.DataFrame(),path='',sheet_name='Sheet1',keys=[],date_cols=[]):
    """
    Appends a dataframe to an existing Excel file,
    keys: Fields that should not be duplicated, the old records are kept
    path: Path to the Excel file
    """
    if df_new.empty:
        return
    if not keys:
        keys=df_new.columns
    df_new=df_new.copy()
    df_new=format_dates(df_new,date_cols)
    if not os.path.exists(path):
        # If the file doesn't exist, save the dataframe as a new Excel file
        save_df(df_new,path,sheet_name=sheet_name,index=False)
        return
    close_xl_if_open(path)
    df=read_excel(path)
    df=format_dates(df,date_cols)
    merged = pd.merge(df_new, df[keys], on=keys, how='left', indicator=True)
    df_new=df_new.iloc[merged[merged['_merge'] == 'left_only'].index]
    df=pd.concat([df,df_new])  
    df_grp=df.groupby(keys).count()
    df_grp=df_grp[df_grp[df_grp.columns[0]]>1]
    if len(df_grp)>0:
        df_grp.reset_index(inplace=True)
        show_popup_message(f"Hay duplicados en el archivo {path} para la llave {keys}: {df_grp.head()[keys]}")
        raise SystemExit()
    save_df(df,path,sheet_name=sheet_name,index=False)

##### Excel management functions
def find_cell_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                return cell.coordinate  # Return cell address (e.g., 'B2')
    return None 

def find_all_cells_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    cells=[]
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                cells.append(cell.coordinate)
    return cells 

def get_cell_properties(cell):
    properties = {}
    # Get the fill color (background)
    fill = cell.fill
    if fill and fill.fgColor and fill.fgColor.type == "rgb":
        rgb = fill.fgColor.rgb  # ARGB format
        properties["background_color"] = f"{rgb[2:]}"  # Skip alpha
    else:
        properties["background_color"] = None
    # Get the font color (text color)
    font_color = cell.font.color
    if font_color and font_color.type == "rgb":
        rgb = font_color.rgb  # ARGB format
        properties["text_color"] = f"{rgb[2:]}"  # Skip alpha
    else:
        properties["text_color"] = 'FFFFFF'
    properties["font_name"]=cell.font.name
    properties["font_size"]=cell.font.size
    # Check if the content is wrapped
    properties["is_wrapped"] = cell.alignment.wrap_text if cell.alignment else False
    # Check if the text is bold
    properties["is_bold"] = cell.font.bold if cell.font else False
    return properties

def format_cell(cell,properties):
    "Apply properties based on the dict of prooperties"
    if properties['background_color']:
        cell.fill = PatternFill(start_color=properties['background_color'], end_color=properties['background_color'], fill_type="solid")
    else:
        cell.fill = PatternFill(fill_type=None)
    cell.font = Font(name=properties['font_name'],size=properties['font_size'],color=properties['text_color'], bold=properties['is_bold'])
    cell.alignment = Alignment(wrap_text=properties['is_wrapped'])

def format_on_change(zip_cols, ws, start_row=1, format1=None, format2=None):
    """
    Formats rows in a worksheet based on changes in values across zipped columns.

    Parameters:
    - zip_cols: zip of multiple columns (e.g., zip(col_info1, col_info2, ...)).
    - worksheet: The active worksheet to apply formatting.
    - start_row: Starting row number (default is 1).
    - format1: First alternating format (default yellow).
    - format2: Second alternating format (default light blue).
    """
    # Default styles
    if format1 is None:
        format1 = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")  # Yellow
    if format2 is None:
        format2 = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")  # Light Blue

    # Initialize tracking variables
    previous_values = None
    current_format = format1

    # Iterate through the zipped columns and their row numbers
    for row_idx, cells_rows in enumerate(zip_cols, start=start_row):
        # Check if the current row values differ from the previous row
        combined_values=""
        for cell in cells_rows:
            combined_values=combined_values+cell.value
        if previous_values != combined_values:
            # Alternate the format
            current_format = format1 if current_format == format2 else format2
            previous_values = combined_values

        # Apply the format to all cells in the current row from the zipped columns
        for col_idx, cell in enumerate(cells_rows):
            # cell = ws.cell(row=row_idx, column=col_idx + 1)
            format_cell(cell, current_format)

def get_column_info(ws, col_name):
    """
    Returns a dictionary with the column cell and the data range for the column.
    """
    col_cell=find_cell_by_text(ws, col_name)
    if not col_cell:
        show_popup_message(f"Column '{col_name}' not found in the sheet.")
        raise SystemExit()
    col_cell=ws[col_cell]
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    data_range=ws[col_cell.offset(1,0).coordinate:ws.cell(ws[last_cell].row,col_cell.offset(1,0).column).coordinate]
    data_range_list=[cell[0] for cell in data_range]
    return {'data_range':data_range_list,
            'col_cell':col_cell}

def set_number_format(ws,col_name,format):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=find_cell_by_text(ws,col_name)
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].number_format = format

def fill_column(ws,col_name,fill):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].fill=fill

def font_column(ws,col_name,font):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].font=font    



def fill_formula(ws,col_name,formula):
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    col_head_address=ws[find_cell_by_text(ws,col_name)].offset(1,0).coordinate
    for row in ws[f"{col_head_address}:{last_cell}"]:
        row[0].value=formula

    
def get_xl_formatting(table_name=None):
    """
    Returns the formatting for the excel file defined in columns and formatting.xlsx
    return example
    {'oor':
    {
        'columna':{
            'head':{properties},
            'data':{properties}
        }
    }
    }
    """
    wb = load_workbook(output_paths['path_xl_format'])
    ws = wb['column_format']
    col_info=get_column_info(ws,'column_name')
    # df_format=pd.DataFrame()
    cell_properties={}
    for cell in col_info['data_range']:
        head_properties=get_cell_properties(cell)
        data_properties=get_cell_properties(cell.offset(0, 1))
        # print(cell.offset(0, -1).value)
        if cell.offset(0, -1).value not in cell_properties:
            cell_properties[cell.offset(0, -1).value]={} #Table name
        if cell.value not in cell_properties[cell.offset(0, -1).value]:
            cell_properties[cell.offset(0, -1).value][cell.value]= {} #Column name
        cell_properties[cell.offset(0, -1).value][cell.value]['head_properties']=head_properties
        cell_properties[cell.offset(0, -1).value][cell.value]['data_properties']=data_properties
        # df_format=pd.concat([df_format,pd.DataFrame([cell_properties])])
    ws = wb['special_format']
    col_info=get_column_info(ws,'format_name')
    cell_properties['special_format']={}
    for cell in col_info['data_range']:
        cell_properties['special_format'][cell.value]=get_cell_properties(cell)

    if table_name:
        cell_properties=cell_properties[table_name]
    return cell_properties  

def get_col_sizes(wb):
    """
    Obtains a dictionary with the column sizes of each sheet in a workbook
    """
    col_sizes={}
    for sheet in wb.sheetnames:
        col_sizes[sheet]=wb[sheet].column_dimensions
    return col_sizes


def apply_col_sizes(wb,col_sizes):
    """
    Applies the column sizes obtained by function get_col_sizes
    """
    for sheet in col_sizes.keys():
        if sheet not in wb.sheetnames:
            continue
        ws=wb[sheet]
        for column_letter in col_sizes[sheet].keys():
            if column_letter==0:
                continue
            ws.column_dimensions[column_letter].width=col_sizes[sheet][column_letter].width
# -------------------------

# Save and load state using pickle
def save_state_pickle(state, filename='folder_state_local_shipments.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(state, f)

def load_state_pickle(filename='folder_state_local_shipments.pkl'):
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        return {"folder_input": None, "folder_output": None, "selections": {}}

def set_opposites(name_a,name_b):
    opposite = {name_a: name_b, name_b: name_a}
    return opposite

    
# Agregar las columnas criticas por archivo
filters = ['Tracker',
           'ELP Master',
           'Shipment transactions',
           'InventoryStageBakup',
           'Bracket',
           'Angle dimming',
           'Visagras']


mandatory_cols={
    'Korrus':[
        'PurchaseOrder',
        'PODate',
        'REF02_ClearTextClause',
        'REF02_CustomerOrderNumber',
        'REF02_CarrierAccount',
        'TermsTypeCode',
        'TermsBasisDateCode',
        'description',
        'Routing',
        'JobName',
        'JobNameNotes',
        'ShipmentMarkings',
        'ShipmentMarkingsNotes',
        'ShipToParty',
        'ShipToAddress1',
        'ShipToAddress2',
        'ShipToCity',
        'ShipToState',
        'ShipToPostalCode',
        'ShipToCountryCode',
        'BillToParty',
        'Address1',
        'Address2',
        'BillToCity',
        'BillToState',
        'BillToPostalCode',
        'BillToCountryCode',
        'Supplier',
        'SupplierAddress1',
        'SupplierAddress2',
        'SupplierCity',
        'SupplierState',
        'SupplierPostalCode',
        'SupplierCountryCode',
        'LineNumber',
        'Quantity',
        'Uom',
        'price',
        'ProductServiceID',
        'StorageLocation',
        'AssignedDropZone'
        ],
    'Tracker':[
        'PO cliente',
        'Modelo',
        'WO\n QTY',
        'START DATE',
        'REPROGRAMACION',
        'FINISH DATE',
        'SHIP DATE'
        ],
    'Shipment transactions':[
        'Part No.',
        'Customer PO#',
        'Date Shipped',
        'Qty. Shipped'
    ],
    'InventoryStageBakup':[
        'Producto',
        'PO',
        'Box Id',
        'Cantidad'
    ],
    'ELP Master':[
        'CUU ship Date', 
        'PN', 
        'PO', 
        'DZ', 
        'BOX ID', 
        'Unit\nQTY']
                }

mandatory_selectors=[
    'Tracker']

families={'TROV':['L','CS-L','CC-L'],
          'RISE':['F','CS-F'],
          'Accesorio':['RISE','M','CBL','LENS','WMA'],
          'DZ':['AssignedDropZone']}

column_mapping={
    'PO':['Customer PO#','PO cliente','PurchaseOrder','PO.'],
    'Modelo':['Producto','ProductServiceID','Part No.','PN','ProductService ID'],
    'Quantity':['WO\n QTY','Cantidad','PO Quantity','QTY','WO\n QTY','Unit\nQTY'],
    'ShipmentDate':['CUU ship Date'],
    'PODate':['EDI Received'],
    'WO':['Work Order'],  
    'SOUS': ['REF02_CustomerOrderNumber']  
}

def standardize_columns(df, column_mapping):
    """Estandariza las columnas de un DataFrame según un diccionario de mapeo."""
    reverse_mapping = {orig: standard for standard, cols in column_mapping.items() for orig in cols}
    return df.rename(columns=reverse_mapping)

def inverse_standardize_columns(df, column_mapping, columns_to_replace):
    """Reverts standard column names to original names based on column_mapping."""
    inverse_mapping = {standard: orig for standard, cols in column_mapping.items() for orig in cols if orig in columns_to_replace}
    return df.rename(columns=inverse_mapping)

def check_mandatory_cols(cols,selector_name, raise_error=True):
    missing_columns = [col for col in mandatory_cols[selector_name] if col not in cols]
    if len(missing_columns)>0:
        show_popup_message(f"No se encontraron las siguientes columnas en el archivo {selector_name}: {missing_columns}")
        if raise_error:
            raise SystemExit()
        return False
    return True
state = load_state_pickle()
if ('folder_output' not in state.keys()):
    state['folder_output']=''

def set_paths(path):
    """
    Set global paths
    """
    global output_paths
    output_paths={}
    output_paths['path_attachments']=os.path.join(path, "attachments")
    output_paths['path_attachments_done']=os.path.join(output_paths['path_attachments'], "Done")
    output_paths['path_korrus_list']=os.path.join(path,'Korrus_list.xlsx')
    output_paths['path_korrus_data']=os.path.join(path,'Korrus_data.xlsx')
    output_paths['path_edi']=os.path.join(path,'EDI Master.xlsx')
    output_paths['path_ship_cust']=os.path.join(path,'Shipped to Cust.xlsx')
    output_paths['path_ship_elp']=os.path.join(path,'Shipped to ELP.xlsx')
    output_paths['path_oor']=os.path.join(path,'OOR Report.xlsx')
    output_paths['path_xl_format']=os.path.join(path,'columns and formatting.xlsx')
    output_paths['path_graph_data']=os.path.join(path,'graph_data.xlsx')



folder_output_button = widgets.Button(description="Folder de salidas:")
if state['folder_output']:
    initialdir=state['folder_output']
    set_paths(initialdir)
else:
    initialdir='Not selected'
folder_output_label = widgets.Label(value=initialdir)

    

def on_output_button_click(b):
    if state['folder_output']:
        initialdir=state['folder_output']
    else:
        initialdir='/'
    selected_dir = select_directory(initialdir=initialdir)
    if selected_dir:
        folder_output_label.value = f"{selected_dir}"
        state['folder_output']=selected_dir
        save_state_pickle(state)
    else:
        folder_output_label.value = "Not selected"
    set_paths(folder_output_label.value)




# Attach the event to the folder_output_button
folder_output_button.on_click(on_output_button_click)

# Create an array of button and label widgets
file_selectors = []
for filter_name in filters:
    # Create button and label
    button = widgets.Button(description=f"{filter_name}:")
    if state.get(filter_name):
        label = widgets.Label(value=f" {state.get(filter_name)}")
    else:
        label = widgets.Label(value=f" Not selected")
    
    # Define the button click event
    def on_button_click(filter_name=filter_name, label=label):
        if (filter_name in state.keys()) and (state[filter_name]):
            initialdir=os.path.dirname(state[filter_name][0])
        else:
            initialdir='/'
        selected_dir = open_file_selection(initialdir=initialdir,filter_name=filter_name)  # Adjust initialdir as needed
        if selected_dir:
            label.value = f" {selected_dir[0]}"
            state[filter_name]=selected_dir[0]
            
            
        else:
            label.value = f" Not selected"
            state[filter_name]=f" Not selected"
        save_state_pickle(state)

    # Attach the event to the button
    button.on_click(lambda b, f=filter_name, l=label: on_button_click(f, l))
    
    # Add the button and label as a horizontal box
    file_selectors.append(widgets.HBox([button, label]))


datepicker_email = widgets.DatePicker(
    description='Fecha mail',
    disabled=False,
    value=date.today()-timedelta(days=1)
)

datepicker_shipments_elp = widgets.DatePicker(
    description='ELP Shipments',
    disabled=False,
    value=date.today()
)


ui = widgets.VBox([datepicker_email,datepicker_shipments_elp]+[widgets.HBox([folder_output_button, folder_output_label])]+file_selectors)
display(ui)



## Explorar outlook
- Obtiene los archivos Korrus adjuntos de las fechas no obtenidas anteriormente

In [221]:
# Obtener los archivos adjuntos de los emails

# Crear folder para adjuntos si no existe

if not os.path.exists(output_paths['path_attachments']):
    os.makedirs(output_paths['path_attachments'])

outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
inbox = outlook.GetDefaultFolder(6)  
date_limit = datepicker_email.value
date_limit=datetime.combine(date_limit, datetime.min.time())
messages = inbox.Items
messages.Sort("[ReceivedTime]", True)  
# df_korrus_list_new=pd.DataFrame(columns=['file_name','received_time','status'])
for message in messages:
    if not hasattr(message, 'ReceivedTime'):
        continue
    received_time = message.ReceivedTime
    received_time = datetime.fromtimestamp(received_time.timestamp())
    if received_time < date_limit:
        break 
    # print(f"Processing email: {message.Subject} (Received: {received_time})")
    if message.Attachments.Count > 0:
        for attachment in message.Attachments:
            # Save the attachment
            attachment_name = attachment.FileName
            attachment_name=f"{os.path.splitext(attachment_name)[0]}_{received_time.strftime('%Y-%m-%d-%H%M%S')}{os.path.splitext(attachment_name)[1]}"
            # if attachment_name in df_korrus_list['file_name'].values:
            #     continue
            if 'KORRUS' not in attachment_name.upper():
                continue
            path_attachment = os.path.join(output_paths['path_attachments'], attachment_name)
            if  not os.path.exists(path_attachment):
                try:
                    attachment.SaveAsFile(path_attachment)
                except Exception as e:
                    print(f"Error saving attachment: {e}")


### Consolidar archivos Korrus
En caso de un error, hay que abrir el archivo Korrus_list.xlsx y eliminar la e (error) del archivo que se corrigio
En caso de requerir reprocesar un archivo, hay que abrir el archivo Korrus_list.xlsx y poner r (reproceso) en el archivo que se va a reprocesar

In [141]:
#Crear la lista de archivos descargados para manejar su integracion. Se procesa lo que esta en el folder sin importar su origen

if not os.path.exists(output_paths['path_attachments_done']):
    os.makedirs(output_paths['path_attachments_done'])

#Status maneja el proceso a realizar en los archivos: r=reprocesar, d=done, e=error
if os.path.exists(output_paths['path_korrus_list']):
    df_korrus_list=pd.read_excel(output_paths['path_korrus_list'])
else:
    df_korrus_list=pd.DataFrame(columns=['file_name','received_time','status'])
close_xl_if_open(output_paths['path_korrus_list'])

files=os.listdir(output_paths['path_attachments'])
df_korrus_list_new=pd.DataFrame(columns=['file_name'],data=files)
df_korrus_list_new['received_time']=df_korrus_list_new['file_name'].str.extract(r'(\d{4}-\d{2}-\d{2}-\d{6})')
df_korrus_list_new=df_korrus_list_new.merge(df_korrus_list,how='outer',on=['file_name','received_time'])
df_korrus_list_new.dropna(subset=['received_time'],inplace=True)
df_korrus_list_new['status'].fillna('',inplace=True)

# Analizar archivos adjuntos
df_korrus_data_new=pd.DataFrame(columns=['origin_file']+mandatory_cols['Korrus'])
for index, row in df_korrus_list_new[df_korrus_list_new['status'].isin(['','r'])].iterrows():
    filepath=os.path.join(output_paths['path_attachments'],row['file_name'])
    if (row['status']=='r') or (not os.path.exists(filepath)):
        filepath=os.path.join(output_paths['path_attachments_done'],row['file_name'])
    if not os.path.exists(filepath):
            df_korrus_list_new.loc[index,'status']='e'
            show_popup_message(f"El archivo {row['file_name']} no existe")
            continue
    if os.path.splitext(os.path.basename(filepath))[1].lower()==".xlsx":
        df = pd.read_excel(filepath)
    elif os.path.splitext(os.path.basename(filepath))[1].lower()==".csv":
        try:
            df = pd.read_csv(filepath, sep='\t', encoding='utf-16')
        except:
            df = pd.read_csv(filepath, sep=',')
    else:
        df_korrus_list_new.loc[index,'status']='e'
        show_popup_message(f"El archivo {row['file_name']} no pudo ser procesado")
        continue
    if check_mandatory_cols(df.columns,selector_name='Korrus',raise_error=False):
        df['origin_file']=row['file_name']
        df_korrus_data_new=pd.concat([df_korrus_data_new,df])
        df_korrus_data_new=df_korrus_data_new[['origin_file']+mandatory_cols['Korrus']]
        df_korrus_list_new.loc[index,'status']='n'
    else:
        df_korrus_list_new.loc[index,'status']='e'

# Consolidar datos en un solo archivo
if os.path.exists(output_paths['path_korrus_data']):
    df_korrus_data=pd.read_excel(output_paths['path_korrus_data'])
else:
    df_korrus_data=pd.DataFrame(columns=['origin_file']+mandatory_cols['Korrus'])
close_xl_if_open(output_paths['path_korrus_data'])
df_korrus_data=df_korrus_data[~df_korrus_data['origin_file'].isin(df_korrus_data_new['origin_file'])]
df_korrus_data_new=pd.concat([df_korrus_data, df_korrus_data_new])

# Mover archivos a la carpeta de archivos procesados
for index,row in df_korrus_list_new[df_korrus_list_new['status']=='n'].iterrows():
    df_korrus_list_new.loc[index,'status']='y'
    if os.path.exists(f'{output_paths['path_attachments']}/{row["file_name"]}'):
        shutil.move(f'{output_paths['path_attachments']}/{row["file_name"]}', f'{output_paths['path_attachments_done']}/{row["file_name"]}')
# Eliminar archivos ya procesados del folder de llegada
df_remove=df_korrus_list_new[(df_korrus_list_new['status']=='y') & (df_korrus_list_new['file_name'].isin(files))]
for index,row in df_remove.iterrows():
    if os.path.exists(os.path.join(output_paths['path_attachments'],row['file_name'])):
        os.remove(os.path.join(output_paths['path_attachments'],row['file_name']))
files=os.listdir(output_paths['path_attachments'])
# Guardar datos y lista de archivos procesados
save_df(df_korrus_list_new,filepath=output_paths['path_korrus_list'],sheet_name='Korrus List',index=False)
save_df(df_korrus_data_new,filepath=output_paths['path_korrus_data'],sheet_name='Korrus Data',index=False)

## Generar reportes
- Hay tres reportes EDI Master, Shipped to Cust, Shipped to ELP
- Si hay archivos seleccionados se integran a estos reportes

In [164]:
# Consolidar reportes 
def set_family(df,column):
    """
    column: name of the column that contains the Model
    """
    for key in families.keys():
        df.loc[df[column].str.startswith(tuple(families[key])), 'Family'] = key
    df['Family'].fillna('Other', inplace=True)
    return df

if not os.path.exists(output_paths['path_xl_format']):
    show_popup_message("No se encuentra el archivo: columns and formatting.xlsx")
    raise SystemExit()

if folder_output_label.value=='Not selected':
    show_popup_message("Seleccione el folder de salida")
    raise SystemExit()
verify_selections(file_selectors)
close_xl_if_open(output_paths['path_oor'])
# Demanda del cliente
if not os.path.exists(output_paths['path_korrus_data']):
    df=pd.DataFrame(columns=[mandatory_cols['Korrus']+['origin_file']])
    save_df(df,filepath=output_paths['path_korrus_data'],sheet_name='Korrus Data',index=False)

df_korrus_data_new=pd.read_excel(output_paths['path_korrus_data'])
df_korrus_data_new=df_korrus_data_new[~df_korrus_data_new['PurchaseOrder'].str.contains('---')]
df_korrus_data_new['PODate']=pd.to_datetime(df_korrus_data_new['PODate'],format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
df_korrus_data_new['LineNumber']=df_korrus_data_new['LineNumber'].astype(int)
key_korrus=['PurchaseOrder','LineNumber','PODate','ProductServiceID','AssignedDropZone']
df_korrus_data_new.drop_duplicates(subset=key_korrus,keep='last',inplace=True)

# Eliminar registros no requeridos
df_korrus_data_new.drop('origin_file',axis=1,inplace=True)
df_korrus_data_new.drop_duplicates(inplace=True)
edi_keys=['PurchaseOrder','PODate','LineNumber','ProductServiceID','AssignedDropZone']
df_korrus_data_new['PODate']=pd.to_datetime(df_korrus_data_new['PODate'])
append_df_to_excel(df_korrus_data_new,output_paths['path_edi'],sheet_name='Edi Master',keys=edi_keys,date_cols=['PODate'])

# Shipment transactions, lo embarcado al cliente
path_ship_cust_new=get_path(file_selectors,'Shipment transactions')
if path_ship_cust_new!='Not selected':
    df_ship_cust_new=pd.read_excel(path_ship_cust_new)
    check_mandatory_cols(df_ship_cust_new.columns,'Shipment transactions')
    df_ship_cust_new=df_ship_cust_new[~df_ship_cust_new['Customer PO#'].isna()]
    # append_df_to_excel(df_ship_cust_new,output_paths['path_ship_cust'],sheet_name='Shipped to Cust',keys=['Part No.', 'Customer PO#', 'Date Shipped'])
    # Shipment transactions is not agregated, it is replaced
    save_df(df_ship_cust_new,output_paths['path_ship_cust'],sheet_name='Shipped to Cust',index=False)

    
# InventoryStage, lo que se embarco a ELP 
path_ship_elp_new=get_path(file_selectors,'InventoryStageBakup')
if path_ship_elp_new!='Not selected':
    df_ship_elp_new=pd.read_excel(path_ship_elp_new)
    check_mandatory_cols(df_ship_elp_new.columns,'InventoryStageBakup')
    df_ship_elp_new.rename({'Producto':'PN','Box Id':'BOX ID','Cantidad':'Quantity'},axis=1,inplace=True)
    df_ship_elp_new=df_ship_elp_new[~df_ship_elp_new['PO'].isna()]
    df_ship_elp_new=df_ship_elp_new[['Cliente','PO','PN','BOX ID','Quantity']].ffill()
    df=df_ship_elp_new['PO'].str.split('|', expand=True)
    df_ship_elp_new['DZ']=''
    if len(df.columns)>1:
        df_ship_elp_new['DZ']=df[1].str.strip()
        df_ship_elp_new['PO']=df[0].str.strip()
    df_ship_elp_new['CUU ship Date']=datepicker_shipments_elp.value
    df_ship_elp_new['CUU ship Date']=pd.to_datetime(df_ship_elp_new['CUU ship Date'])
    df_ship_elp_new=standardize_columns(df_ship_elp_new,column_mapping)
    append_df_to_excel(df_ship_elp_new,output_paths['path_ship_elp'],sheet_name='Shipped to ELP',keys=['PO','Modelo','BOX ID'])

# Integrar ELP Master si esta saleccionado (incluye hojas: EDI master y Shipment to ELP)
path_ship_elp_master=get_path(file_selectors,'ELP Master')
if path_ship_elp_master!='Not selected':
    df_ship_elp_master=read_excel(path_ship_elp_master,sheet_name='Shipment to ELP')
    df_ship_elp_master.rename({'PO.':'PO'},axis=1,inplace=True)
    df_ship_elp_master=df_ship_elp_master[mandatory_cols['ELP Master']]
    df_ship_elp_master=standardize_columns(df_ship_elp_master,column_mapping)
    append_df_to_excel(df_ship_elp_master,output_paths['path_ship_elp'],sheet_name='Shipped to ELP',keys=['PO','Modelo','BOX ID'])
    df_edi_master=read_excel(path_ship_elp_master,sheet_name='EDI Master')
    df_edi_master=inverse_standardize_columns(df_edi_master,column_mapping,['PurchaseOrder','ProductServiceID'])
    df_edi_master.rename({'ProductService ID':'ProductServiceID'},axis=1,inplace=True)
    append_df_to_excel(df_edi_master,output_paths['path_edi'],sheet_name='Edi Master',keys=edi_keys,date_cols=['PODate'])

In [165]:
# Funcion, asignar cantidades
def assign_quantities(df_pos, df_to_assign, additional_fields=[]):
    """
    Asigna las cantidades del segundo dataframe al primero, respetando los límites indicados en el campo Quantity del primer dataframe.
    df_pos: requiere las columnas 'PO', 'Modelo', 'Quantity' y 'Assigned'
    df_to_assign: requiere las columnas 'PO', 'Modelo', 'Quantity'
    """
    df_pos = df_pos.copy()
    df_to_assign=df_to_assign.copy()
    df_pos['Assigned'] = 0

    # Create a new DataFrame to track WorkOrder assignments
    df_assignments = []

    # Assign produced quantities respecting the limits
    for ln_index,ln_row in df_to_assign[['PO','Modelo']].drop_duplicates().iterrows():
        df_assign_subset = df_to_assign[(df_to_assign['PO'] == ln_row['PO']) & (df_to_assign['Modelo'] == ln_row['Modelo'])].copy()
        df_assign_subset['Remaining'] = df_assign_subset['Quantity']

        for wo_index, subset_row in df_assign_subset.iterrows():
            remaining_produced = subset_row['Remaining']

            for index, row in df_pos[(df_pos['PO'] == ln_row['PO']) & (df_pos['Modelo'] == ln_row['Modelo'])].iterrows():
                if remaining_produced <= 0:
                    break

                available_space = row['Quantity'] - df_pos.at[index, 'Assigned']
                assignable = min(available_space, remaining_produced)

                if assignable > 0:
                    df_pos.at[index, 'Assigned'] += assignable
                    remaining_produced -= assignable

                    # Track the assignment
                    line_assignment ={
                        # 'WO': subset_row['WO'],
                        'PO': row['PO'],
                        'Modelo': row['Modelo'],
                        'AvailQuantity': subset_row['Quantity'],
                        'LineNumber': row['LineNumber'],
                        'Assigned_Quantity': assignable
                        }
                    # Other fields if exist
                    if additional_fields:
                        for field in additional_fields:
                            if field in subset_row:
                                line_assignment[field]=subset_row[field]
                    df_assignments.append(line_assignment)

    # Create the assignments DataFrame
    df_assignments = pd.DataFrame(df_assignments)
    assigned={'df_pos':df_pos,'df_assignments':df_assignments}
    return assigned


In [166]:
# Reporte de work orders, que se encuentra en proceso de produccion
path_tracker=get_path(file_selectors,'Tracker')
df_wo_wb=pd.read_excel(path_tracker,sheet_name=None)
df_wo=pd.DataFrame()
for sheet in df_wo_wb.keys():
    if 'Plan de produccion' in sheet:
        df_wo=pd.concat([df_wo,df_wo_wb[sheet]])
        df_wo.dropna(subset=['PO cliente','Modelo'],inplace=True)
check_mandatory_cols(df_wo.columns,'Tracker')
df_wo=standardize_columns(df_wo,column_mapping)
df_wo['Modelo']=df_wo['Modelo'].str.upper()
#% Edi
if not os.path.exists(output_paths['path_edi']):
    show_popup_message('El EDI Master no se ha generado, integre archivos Korrus o seleccione ELP master log')
    raise SystemExit()
df_edi=read_excel(path=output_paths['path_edi'])
df_edi=standardize_columns(df_edi,column_mapping)
df_edi['Modelo']=df_edi['Modelo'].str.upper()

In [167]:
# Procesar envios al cliente eliminando cantidades negativas. Se conserva la fecha mas nueva de envio.
if not os.path.exists(output_paths['path_ship_cust']):
    show_popup_message('No hay envios al paso, integre al menos un Shipment Transactions')
    raise SystemExit()
df_ship_cust=read_excel(path=output_paths['path_ship_cust'])


df_ship_cust=standardize_columns(df_ship_cust,column_mapping)
df_ship_cust['Modelo']=df_ship_cust['Modelo'].str.upper()
df_ship_cust_grp=df_ship_cust.groupby(['PO','Modelo']).sum('Qty. Shipped').reset_index()
df_ship_cust_grp=df_ship_cust_grp[['PO','Modelo','Qty. Shipped']]
df_ship_cust_dates=df_ship_cust.sort_values(['PO','Modelo','Date Shipped']).drop_duplicates(subset=['PO','Modelo'],keep='last')
df_ship_cust_dates.drop('Qty. Shipped',axis=1,inplace=True)
df_ship_cust_dates=df_ship_cust_dates.merge(df_ship_cust_grp,on=['PO','Modelo'])
df_ship_cust_dates=df_ship_cust_dates[['PO','Modelo','Qty. Shipped','Date Shipped']]
df_ship_cust_dates.rename(columns={'Date Shipped':'Date Shipped ELP','Qty. Shipped':'Quantity'},inplace=True)

In [168]:
# Envios a ELP
df_ship_elp=read_excel(path=output_paths['path_ship_elp'])
df_ship_elp=standardize_columns(df_ship_elp,column_mapping)
df_ship_elp['Modelo']=df_ship_elp['Modelo'].str.upper()
df_ship_elp['PO']=df_ship_elp['PO'].str.strip()
df_ship_elp['Modelo']=df_ship_elp['Modelo'].str.strip()
df_ship_elp['Quantity'].fillna(0,inplace=True)

df_ship_elp_grp=df_ship_elp.groupby(['PO','Modelo','ShipmentDate']).sum('Quantity').reset_index()
df_ship_elp_grp=df_ship_elp_grp[['PO','Modelo','Quantity','ShipmentDate']]

In [169]:
# Merge EDI con work orders y embarques
assignments_wo=assign_quantities(df_pos=df_edi,df_to_assign=df_wo,additional_fields=['WO','START DATE','FINISH DATE','REPROGRAMACION'])
df_edi_combined=assignments_wo['df_pos']
df_edi_combined.rename({'Assigned':'WO Qty'},axis=1,inplace=True)
assignments_shp_cust=assign_quantities(df_pos=df_edi_combined,df_to_assign=df_ship_cust_dates,additional_fields=['Date Shipped ELP'])
df_edi_combined=assignments_shp_cust['df_pos']
df_edi_combined.rename({'Assigned':'Shipped to Cust'},axis=1,inplace=True)
assignments_shp_elp=assign_quantities(df_pos=df_edi_combined,df_to_assign=df_ship_elp,additional_fields=['ShipmentDate'])
df_edi_combined=assignments_shp_elp['df_pos']
df_edi_combined.rename({'Assigned':'Shipped to Elp'},axis=1,inplace=True)

### OOR Report

In [170]:
# Add work order ifo to EDI, only the record with last finish date, the detail is in Work orders sheet
df_assigned_wo = assignments_wo['df_assignments']
if len(df_assigned_wo)==0:
    df_assigned_wo=pd.DataFrame(columns=['PO','Modelo','LineNumber','WO', 'START DATE', 'FINISH DATE'])
df_assigned_wo=df_assigned_wo.sort_values(['FINISH DATE'])
df_assigned_wo.drop_duplicates(['PO','Modelo','LineNumber'],keep='last',inplace=True)
df_edi_combined=df_edi_combined.merge(df_assigned_wo[['PO','Modelo','LineNumber','WO', 'START DATE', 'FINISH DATE']],how='left',on=['PO','Modelo','LineNumber'])

In [171]:
# Set production, and shipped STATUS
df_edi_combined[['Quantity','WO Qty','Shipped to Elp','Shipped to Cust']].fillna(0,inplace=True)
df_po_status=df_edi_combined.groupby(['PO']).sum(['Quantity','WO Qty','Shipped to Elp','Shipped to Cust'])
df_po_status.reset_index(inplace=True)
df_po_status=df_po_status[['PO','Quantity','WO Qty','Shipped to Elp','Shipped to Cust']]
df_po_status['remaining']=df_po_status['Quantity']-df_po_status['WO Qty']
df_po_status.loc[df_po_status['remaining']<=0,'Production status']='PO Complete'
df_po_status['remaining']=df_po_status['Quantity']-df_po_status['Shipped to Elp']
df_po_status.loc[df_po_status['remaining']<=0,'Status']='Shipped to Elp'
df_po_status['remaining']=df_po_status['Quantity']-df_po_status['Shipped to Cust']
df_po_status.loc[df_po_status['remaining']<=0,'Status']='Shipped to Cust'
df_edi_combined=df_edi_combined.merge(df_po_status[['PO','Production status','Status']],on=['PO'],how='left')
df_edi_combined.sort_values(['PO','Modelo','LineNumber','PODate'],inplace=True)
df_edi_combined.reset_index(drop=True,inplace=True)
df_edi_combined['Balance to ship']=df_edi_combined['Quantity']-df_edi_combined['Shipped to Elp']


In [172]:
# Read customer shipments and shipments to elp
df_ship_cust=read_excel(output_paths['path_ship_cust'])
df_ship_cust=standardize_columns(df_ship_cust,column_mapping)
df_ship_cust['Modelo']=df_ship_cust['Modelo'].str.upper()
df_ship_cust.sort_values(['PO','Modelo','Date Shipped'],inplace=True)
df_ship_cust.reset_index(drop=True,inplace=True)
df_ship_elp=read_excel(output_paths['path_ship_elp'])
df_ship_elp=standardize_columns(df_ship_elp,column_mapping)
df_ship_elp['Modelo']=df_ship_elp['Modelo'].str.upper()
df_ship_elp.sort_values(['PO','Modelo','ShipmentDate'],inplace=True)
df_ship_elp.reset_index(drop=True,inplace=True)

In [173]:
# Get indexes to use in hyperlinks
def get_df_idx(df,idx_cols,idx_name='idx'):
    df_idx=df.reset_index()
    df_idx.rename({'index':idx_name},axis=1,inplace=True)
    df_idx=df_idx.drop_duplicates(idx_cols,keep='first')
    df_idx=df_idx[idx_cols+[idx_name]]
    return df_idx

df_wo_idx=get_df_idx(df_wo,idx_cols=['PO','Modelo'],idx_name='idx_wo')
df_ship_elp_idx=get_df_idx(df_ship_elp,idx_cols=['PO','Modelo'],idx_name='idx_elp')
df_ship_cust_idx=get_df_idx(df_ship_cust,idx_cols=['PO','Modelo'],idx_name='idx_cust')
df_edi_combined=df_edi_combined.merge(df_wo_idx,how='left',on=['PO','Modelo'])
df_edi_combined=df_edi_combined.merge(df_ship_elp_idx,how='left',on=['PO','Modelo'])
df_edi_combined=df_edi_combined.merge(df_ship_cust_idx,how='left',on=['PO','Modelo'])
df_edi_combined['WO']=df_edi_combined['WO'].fillna(0).astype(int)

In [174]:
# Generate hyperlinks
def set_hyperlink(df,sheet_name,col_name,idx_name):
    """
    sheet_name: Destination sheet of the hyperlink
    col_name: Column name where the hyperlink will be inserted
    idx_name: Column name where the index is located
    """
    df=df.copy()
    idx=~df[idx_name].isnull()
    df.loc[idx,col_name]=f"=HYPERLINK(\"#'{sheet_name}'!A"+(df.loc[idx,idx_name]+2).astype(int).astype(str)+"\","+(df.loc[idx,col_name]).astype(str)+")"
    return df


df_edi_combined=set_hyperlink(df_edi_combined,sheet_name='WorkOrder Detail',col_name='WO',idx_name='idx_wo')
df_edi_combined=set_hyperlink(df_edi_combined,sheet_name='Shipped to Cust',col_name='Shipped to Cust',idx_name='idx_cust')
df_edi_combined=set_hyperlink(df_edi_combined,sheet_name='Shipped to Elp',col_name='Shipped to Elp',idx_name='idx_elp')

In [175]:
# Rename clumns
df_oor=inverse_standardize_columns(df_edi_combined,column_mapping,['EDI Received','PurchaseOrder','ProductServiceID','PO Quantity','Work Order'])
df_oor=set_family(df_oor,'ProductServiceID')
df_oor=df_oor[
    ['EDI Received',
 'SOUS',
 'PurchaseOrder',
 'LineNumber',
 'PO Quantity',
 'ProductServiceID',
 'AssignedDropZone',
 'Family',
 'Work Order',
 'WO Qty',
 'START DATE',
 'FINISH DATE',
 'Production status',
 'Status',
 'Shipped to Elp',
 'Shipped to Cust',
 'Balance to ship'
 ]
]
df_oor.sort_values(['Family','PurchaseOrder','ProductServiceID','LineNumber'],inplace=True)

### Formato de excel

In [176]:
# Format excel
def move_columns_to_front(df, columns):
    """
    Move specified columns to the beginning of the DataFrame.
    """
    columns = [col for col in columns if col in df.columns]
    remaining_columns = [col for col in df.columns if col not in columns]
    return df[columns + remaining_columns]

close_xl_if_open(output_paths['path_oor'])
df_oor=format_dates(df_oor,['START DATE','FINISH DATE'])
df_oor.loc[df_oor['AssignedDropZone']=='','AssignedDropZone']='NULL'
df_oor['AssignedDropZone'].fillna('NULL',inplace=True)
df_wo=format_dates(df_wo,['Date','START DATE', 'FINISH DATE', 'REPROGRAMACION','SHIP DATE'])
df_wo=move_columns_to_front(df_wo,['PO','Modelo','WO','Quantity','START DATE', 'FINISH DATE', 'REPROGRAMACION','SHIP DATE'])
df_ship_elp.loc[df_ship_elp['DZ']=='','DZ']='NULL'
df_ship_elp['DZ'].fillna('NULL',inplace=True)
df_ship_elp=format_dates(df_ship_elp,['ShipmentDate'])
df_ship_elp=move_columns_to_front(df_ship_elp,['PO','Modelo','ShipmentDate','Quantity'])
df_ship_cust=format_dates(df_ship_cust,['Date Shipped'])
df_ship_cust=move_columns_to_front(df_ship_cust,['PO','Modelo','Date Shipped','Qty. Shipped'])
wb_oor={}
wb_oor['OOR']=df_oor
wb_oor['WorkOrder Detail']=df_wo
wb_oor['Shipped to Elp']=df_ship_elp
wb_oor['Shipped to Cust']=df_ship_cust
col_sizes={}
if os.path.exists(output_paths['path_oor']):
    wb_oor=preserve_info(wb_oor,output_paths['path_oor'])
    # Save column sizes
    wb = load_workbook(output_paths['path_oor'])
    col_sizes=get_col_sizes(wb)
save_df_multiple(wb_oor,output_paths['path_oor'],index=False)
wb = load_workbook(output_paths['path_oor'])
ws = wb['OOR']
dict_report_formats=get_xl_formatting(os.path.basename(output_paths['path_oor']))
for key in dict_report_formats.keys():
    col_name=key
    if not find_cell_by_text(ws,col_name):
        continue
    col_properties=dict_report_formats[key]
    # Header formatting
    head_properties=col_properties['head_properties']
    col_info=get_column_info(ws,col_name)
    format_cell(col_info['col_cell'],head_properties)
    # Data formatting
    data_properties=col_properties['data_properties']
    for cell in col_info['data_range']:
        # cell.font = Font(name=data_properties['font_name'],size=data_properties['font_size'],color=data_properties['text_color'], bold=data_properties['is_bold'])
        format_cell(cell,data_properties)
# Format row when key values change
dict_special_formats=get_xl_formatting('special_format')

col_info1=get_column_info(ws,'PurchaseOrder')['data_range']
col_info2=get_column_info(ws,'ProductServiceID')['data_range']
format_on_change(zip(col_info1,col_info2),ws,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])
# Autofilter and freeze pane
ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = 'A2'

ws=wb['WorkOrder Detail']
col_info1=get_column_info(ws,'PO')['data_range']
col_info2=get_column_info(ws,'Modelo')['data_range']
format_on_change(zip(col_info1,col_info2),ws,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])
# Autofilter and freeze pane
ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = 'A2'

ws=wb['Shipped to Elp']
col_info1=get_column_info(ws,'PO')['data_range']
col_info2=get_column_info(ws,'Modelo')['data_range']
format_on_change(zip(col_info1,col_info2),ws,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])
# Autofilter and freeze pane
ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = 'A2'

ws=wb['Shipped to Cust']
col_info1=get_column_info(ws,'PO')['data_range']
col_info2=get_column_info(ws,'Modelo')['data_range']
format_on_change(zip(col_info1,col_info2),ws,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])
# Autofilter and freeze pane
ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = 'A2'
# Apply column sizes
if col_sizes:
    apply_col_sizes(wb,col_sizes)
save_wb(wb,output_paths['path_oor'])
os.startfile(output_paths['path_oor'])

In [178]:
#Datos para graficas
def get_df_category(df,category_name):
    """
    Returns a dataframe with category, date, quantity
    """
    df=df.copy()
    column_mapping_1={'Date':['Date Shipped','ShipmentDate','PODate'],
                    'Quantity':['Qty. Shipped']}
    df=standardize_columns(df,column_mapping_1)
    df['Category']=category_name
    df=df[['Category','Date','Quantity']]
    df=df.groupby(['Category','Date']).sum()
    df.reset_index(inplace=True)
    df['Cum Sum']=df['Quantity'].cumsum()
    return df

df=pd.DataFrame()
df_cumul=get_df_category(df_edi,'EDI')
df=get_df_category(df_ship_cust,'Shipped Elp')
if not df.empty:
    df_cumul=pd.concat([df_cumul,df])
df=get_df_category(df_ship_elp,'Shipped CUU')
if not df.empty:
    df_cumul=pd.concat([df_cumul,df])
save_df(df_cumul,output_paths['path_graph_data'],sheet_name='Date totals',index=False)


# Yield reports

In [144]:
#Yield reports
def map_yield_report(df):
    df_yield=df.copy()
    df_yield.columns=df_yield.iloc[8].fillna('NA').to_list()
    df['Yield Reports'].fillna('',inplace=True)
    date_from=df[df['Yield Reports'].str.contains('From')].iloc[0]['Yield Reports']
    date_to=df[df['Yield Reports'].str.contains('From')].iloc[0]['Unnamed: 5']
    df_yield['date_from']=date_from
    df_yield['date_to']=date_to
    df_yield=df_yield[(~df_yield['Part Number'].isna()) & (df_yield['Part Number']!='Part Number')]
    df_yield['date_from']=df_yield['date_from'].str.replace('From: ','')
    df_yield['date_to']=df_yield['date_to'].str.replace('To: ','')
    df_yield['date_from']=pd.to_datetime(df_yield['date_from'])
    df_yield['date_to']=pd.to_datetime(df_yield['date_to'])
    df_yield.drop('NA',axis=1,inplace=True)
    df_yield.fillna(0,inplace=True)
    df_yield['month']=df_yield['date_from'].dt.strftime('%Y-%m')
    return df_yield

path_bracket=get_path(file_selectors,'Bracket')
if path_bracket!='Not selected':
    df=read_excel(path_bracket)
    df_bracket=map_yield_report(df)

path_angle=get_path(file_selectors,'Angle dimming')
if path_angle!='Not selected':
    df=read_excel(path_angle)
    df_angle=map_yield_report(df)

path_visagras=get_path(file_selectors,'Visagras')
if path_visagras!='Not selected':
    df=read_excel(path_visagras)
    df_visagras=map_yield_report(df)


# Start panel

In [ ]:
# Start panel
import subprocess
p = subprocess.Popen(["panel", "serve", "panel.ipynb"])

# Stop panel

In [ ]:
#Stop panel
p.terminate()